In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Read the dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

df = pd.read_csv(f"{path}/Q1_data.csv")

In [ ]:
# Task 2: Inspect the first few rows
df.head()

In [ ]:
# Task 3: Display dataset information
df.info()

In [ ]:
# Task 4: Show statistical description
df.describe()

In [ ]:
# Task 5: Plot the target distribution
plt.figure(figsize=(8, 5))
plt.hist(df['Delivery_Time'], bins=30, edgecolor='black')
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Drop the 'Order_ID' column
df = df.drop('Order_ID', axis=1)

In [ ]:
# Task 2: Handle missing values
print(df.isnull().sum())

for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col] = df[col].fillna(df[col].mode()[0])
        else:
            df[col] = df[col].fillna(df[col].median())

In [ ]:
# Task 3: Check and remove duplicates
print(f"Duplicates before: {df.duplicated().sum()}")
df = df.drop_duplicates()
print(f"Duplicates after: {df.duplicated().sum()}")

In [ ]:
# Task 4: Encode categorical variables (One Hot Encoding)
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [ ]:
# Task 5: Apply feature scaling (StandardScaler)
from sklearn.preprocessing import StandardScaler

X = df.drop('Delivery_Time', axis=1)
y = df['Delivery_Time']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)

In [ ]:
# Task 6: Write your code here:

In [ ]:
# Task 1: Write your code here:

In [ ]:
# Task 2, 3, 4, 5: KFold, RandomForest, MAE, Print averaged score
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []
model = None

for train_idx, val_idx in kfold.split(X_scaled):
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = RandomForestRegressor(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)
    mae_scores.append(mae)

print(f"MAE Scores per fold: {mae_scores}")
print(f"Average MAE: {np.mean(mae_scores):.4f}")

In [ ]:
# Task 1: Plot feature importance
feature_importance = model.feature_importances_
feature_names = X_scaled.columns

sorted_idx = np.argsort(feature_importance)

plt.figure(figsize=(10, 8))
plt.barh(range(len(sorted_idx)), feature_importance[sorted_idx])
plt.yticks(range(len(sorted_idx)), feature_names[sorted_idx])
plt.xlabel('Feature Importance')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Plot predicted delivery time histogram
model_final = RandomForestRegressor(n_estimators=100, random_state=42)
model_final.fit(X_scaled, y)
y_pred_all = model_final.predict(X_scaled)

plt.figure(figsize=(8, 5))
plt.hist(y_pred_all, bins=30, edgecolor='black', alpha=0.7)
plt.title('Predicted Delivery Time Distribution')
plt.xlabel('Predicted Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
ensemble_mae_scores = []

for train_idx, val_idx in kfold.split(X_scaled):
    X_train, X_val = X_scaled.iloc[train_idx], X_scaled.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Model 1: RandomForest
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    rf_pred = rf_model.predict(X_val)

    # Model 2: CatBoost
    cb_model = CatBoostRegressor(iterations=100, random_state=42, verbose=0)
    cb_model.fit(X_train, y_train)
    cb_pred = cb_model.predict(X_val)

    # Average predictions
    ensemble_pred = (rf_pred + cb_pred) / 2

    mae = mean_absolute_error(y_val, ensemble_pred)
    ensemble_mae_scores.append(mae)

print(f"Ensemble MAE Scores per fold: {ensemble_mae_scores}")
print(f"Ensemble Average MAE: {np.mean(ensemble_mae_scores):.4f}")